# SteelFlow: delivery, dispatch, and freight EDA

This notebook validates the late-delivery classification target and the freight-cost regression target, including class balance, time drift, and feature availability.

In [ ]:
from pathlib import Path
import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = ROOT / "data"
con = duckdb.connect()
sns.set_theme(style="whitegrid")


In [ ]:
late = con.execute(f"SELECT * FROM read_parquet('{(DATA/'dispatch_delay_model_dataset.parquet').as_posix()}')").fetchdf()
late['dispatch_date'] = pd.to_datetime(late['dispatch_date'])
late['prediction_moment'] = pd.to_datetime(late['prediction_moment'])
late.groupby('split').agg(rows=('shipment_id','size'), late_rate=('target_is_late','mean'), min_date=('dispatch_date','min'), max_date=('dispatch_date','max'))

In [ ]:
plt.figure(figsize=(8, 4)); sns.barplot(data=late, x='split', y='target_is_late', errorbar=None); plt.title('Late-delivery rate by split'); plt.ylabel('Rate'); plt.show()

In [ ]:
late.groupby('route_type')['target_is_late'].agg(['count','mean']).sort_values('mean', ascending=False).head(15)

In [ ]:
freight = con.execute(f"SELECT * FROM read_parquet('{(DATA/'freight_cost_model_dataset.parquet').as_posix()}')").fetchdf()
freight['prediction_moment'] = pd.to_datetime(freight['prediction_moment'])
freight.groupby('split')['target_actual_billed_freight_cost_inr_ex_gst'].describe()

In [ ]:
# Target leakage guardrails: prediction_moment precedes outcome by construction; exclude actual outcome fields from features.
late.filter(regex='target|actual|arrival|delay').columns.tolist()

## Modeling handoff

The supplied dispatch and freight marts are already aligned at shipment grain and include chronological split labels. They can be modeled directly after dropping target and post-outcome columns.